## Experiments

In [15]:
from llama_cpp import Llama

In [2]:
llm = Llama(model_path="/home/jobin/Projects/multimodal_rag_summarization/gguf/Meta-Llama-3-8B-Instruct.Q6_K.gguf", chat_format="chatml")

llama_model_loader: loaded meta data with 22 key-value pairs and 291 tensors from /home/jobin/Projects/multimodal_rag_summarization/gguf/Meta-Llama-3-8B-Instruct.Q6_K.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = .
llama_model_loader: - kv   2:                           llama.vocab_size u32              = 128256
llama_model_loader: - kv   3:                       llama.context_length u32              = 8192
llama_model_loader: - kv   4:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   5:                          llama.block_count u32              = 32
llama_model_loader: - kv   6:                  llama.feed_forward_length u32              = 14336
llama_model_loader: - kv 

In [24]:
def summarize_chunk(chunk: str):
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": """You are a helpful assistant that outputs in JSON. You are provided with a partial transcript from an interview. From this, generate a short summary in 50 words. Directly start the summary. Don't preface it with phrases like "Here is a summary...". Start the summary with "In this part, ".""",
            },
            {"role": "user", "content": chunk},
        ],
        response_format={
            "type": "json_object",
            "schema": {
                "type": "object",
                "properties": {"summary": {"type": "string"}},
                "required": ["summary"],
            },
        },
        temperature=0.0,
    )
    return response

def bulletify_summary(summary: str):
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": """You are a helpful assistant that outputs in JSON. You are an expert at converting text into short points. Convert the given text into short points and a title. Don't add escape characters. Directly list the points and the title, don't add additional text before or after it.""",
            },
            {"role": "user", "content": summary},
        ],
        response_format={
            "type": "json_object",
            "schema": {
                "type": "object",
                "properties": {"title": {"type": "string"},
                            "bullets": {"type": "array", "items": {"type": "string"}}
                            },
                "required": ["title", "bullets"],
            },
        },
        temperature=0.0
    )
    return response


In [25]:
summary = "In this part, Mark Cuban shares his experience as an investor in Travis Kalanick's company Red Swish, which later became Uber. He discusses how he saw potential in the idea but was concerned about the challenges of fighting incumbent taxi commissions. He also reflects on the lessons learned from the story, including the importance of marketing and being prepared to take risks."

In [27]:
response = bulletify_summary(summary)

from_string grammar:
bullets ::= [[] space bullets_6 []] space 
space ::= space_12 
bullets_2 ::= string bullets_5 
string ::= ["] string_13 ["] space 
bullets_4 ::= [,] space string 
bullets_5 ::= bullets_4 bullets_5 | 
bullets_6 ::= bullets_2 | 
bullets-kv ::= ["] [b] [u] [l] [l] [e] [t] [s] ["] space [:] space bullets 
char ::= [^"\] | [\] char_9 
char_9 ::= ["\/bfnrt] | [u] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] [0-9a-fA-F] 
root ::= [{] space title-kv [,] space bullets-kv [}] space 
title-kv ::= ["] [t] [i] [t] [l] [e] ["] space [:] space string 
space_12 ::= [ ] | 
string_13 ::= char string_13 | 

Llama.generate: prefix-match hit

llama_print_timings:        load time =    4662.78 ms
llama_print_timings:      sample time =    3059.79 ms /    60 runs   (   51.00 ms per token,    19.61 tokens per second)
llama_print_timings: prompt eval time =       0.00 ms /     1 tokens (    0.00 ms per token,      inf tokens per second)
llama_print_timings:        eval time =    9612.43 ms /    60 

In [33]:
response["choices"][0]["message"]["content"]

'{"title": "Lessons Learned from Red Swish", "bullets": ["Mark Cuban saw potential in Travis Kalanick\'s idea for Red Swish", "He was concerned about fighting incumbent taxi commissions", "Importance of marketing", "Need to be prepared to take risks"] }'

## For Integration

In [1]:
from llama_cpp import Llama
import json

In [2]:
def load_llm(gguf_path: str) -> Llama:
    llm = Llama(model_path=gguf_path, chat_format="chatml", verbose=False)
    return llm

def llamacpp_chat(llm, system_prompt: str, user_prompt: str, schema: dict) -> str:
    response = llm.create_chat_completion(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_object",
            "schema": schema,
        },
        temperature=0.0,
    )
    return response["choices"][0]["message"]["content"]


BULLETIZATION_SYSTEM_PROMPT = """You are a helpful assistant that outputs in JSON. You are an expert at converting text into short points. Convert the given text into short points and a title. Don't add escape characters. Directly list the points and the title, don't add additional text before or after it."""
BULLETIZATION_SCHEMA = {
                            "type": "object",
                            "properties": {"title": {"type": "string"},
                                            "bullets": {"type": "array", "items": {"type": "string"}}
                                            },
                            "required": ["title", "bullets"],
                        }

def bulletize_summary(summary: str, llm: Llama) -> str:
    response_text = llamacpp_chat(llm, 
                                  system_prompt=BULLETIZATION_SYSTEM_PROMPT, 
                                  user_prompt=summary,
                                  schema=BULLETIZATION_SCHEMA)
    return json.loads(response_text)

In [3]:
summary = '''In this part, Mark Cuban shares his entrepreneurial journey, highlighting the importance of preparation and confidence in starting a business. He emphasizes that people often get stuck in the idea phase, fearing the risk involved in leaving their comfort zone. Cuban also touches on the American Dream, suggesting that it's not just about having an idea but being willing to take calculated risks and put in the work to make it happen.'''

In [5]:
llm = load_llm(gguf_path="/home/jobin/Projects/transcript_summarizer/gguf/Meta-Llama-3-8B-Instruct.Q6_K.gguf")

In [6]:
summary_json = bulletize_summary(summary, llm)

In [7]:
summary_json

{'title': "Mark Cuban's Entrepreneurial Journey",
 'bullets': ['Mark Cuban emphasizes the importance of preparation and confidence in starting a business.',
  'People often get stuck in the idea phase, fearing the risk involved in leaving their comfort zone.',
  'The American Dream is not just about having an idea but being willing to take calculated risks and put in the work to make it happen.']}